# Delegator

In [4]:
import pandas as pd
import statsmodels.api as sm
import numpy as np
from stargazer.stargazer import Stargazer

pd.options.display.max_rows = 4000
dg = pd.read_excel('delegator_pilot.xlsx')

#dg = dg[dg['attention']==0.5].reset_index()

dg['female'] = np.nan
dg.loc[dg['Sex']=='Female', 'female'] = 1
dg.loc[dg['Sex']=='Male', 'female'] = 0

In [7]:
dg.head()

,Unnamed: 0,participant.id_in_session,participant.code,participant.overall_score,participant.relevant_round,session.code,bad_outcome_eva,good_outcome_eva,bad_outcome_del,good_outcome_del,...,Age,Sex,Ethnicity simplified,Country of birth,Country of residence,Nationality,Language,Student status,Employment status,female
0,0,1,up0txcxh,3,"{'success': False, 'pic_id': 'r9', 'guess': 14...",ecqu1f6k,1,5,2,4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2,fbckwczw,2,"{'success': False, 'pic_id': 'r7', 'guess': 17...",ecqu1f6k,1,5,2,4,...,32.0,Female,White,Germany,United Kingdom,United Kingdom,English,Yes,Full-Time,1.0
2,2,1,j6qj5zjz,4,"{'success': True, 'pic_id': 'r5', 'guess': 168...",1i2isl7g,1,5,2,4,...,31.0,Male,White,United Kingdom,United Kingdom,United Kingdom,English,No,Full-Time,0.0
3,3,2,k1dav4xq,4,"{'success': True, 'pic_id': 'r4', 'guess': 129...",1i2isl7g,1,5,2,4,...,50.0,Female,White,United Kingdom,United Kingdom,United Kingdom,English,No,Full-Time,1.0
4,4,4,cidrzt8m,1,"{'success': False, 'pic_id': 'r4', 'guess': 11...",1i2isl7g,1,5,2,4,...,56.0,Female,White,United Kingdom,United Kingdom,United Kingdom,English,No,Full-Time,1.0


#### Performance at the task

In [4]:
#overall score
print(dg['participant.overall_score'].value_counts().sort_index())
dg['participant.overall_score'].describe()

1    5
2    6
3    7
4    7
5    1
6    1
Name: participant.overall_score, dtype: int64


count    27.000000
mean      2.851852
std       1.321529
min       1.000000
25%       2.000000
50%       3.000000
75%       4.000000
max       6.000000
Name: participant.overall_score, dtype: float64

In [37]:
#performance by task
for i in range(10):
    if i == 0:
        p_by_task = pd.DataFrame(dg['success_r' + str(i+1)].describe())
    else:
        p_by_task['success_r' + str(i+1)] = dg['success_r' + str(i+1)].describe()

p_by_task

,success_r1,success_r2,success_r3,success_r4,success_r5,success_r6,success_r7,success_r8,success_r9,success_r10
count,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000,27.000000
mean,0.333333,0.222222,0.407407,0.407407,0.407407,0.370370,0.111111,0.111111,0.037037,0.444444
std,0.480384,0.423659,0.500712,0.500712,0.500712,0.492103,0.320256,0.320256,0.192450,0.506370
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


#### Attention checks

In [38]:
#share of people setting slider to 207
sum([1 for i in dg['test_slider'] if i==207])/dg.shape[0]

0.3333333333333333

In [39]:
sum([1 for i in dg['attention'] if i==0.5])/25

0.68

In [40]:
sum([1 for i in dg['error_confidence'] if i==0])/dg.shape[0]

0.7407407407407407

In [41]:
sum([1 for i in dg['error_difficulty'] if i==0])/dg.shape[0]

1.0

In [42]:
dg['pass_att1'] = 0
dg.loc[dg['test_slider']==207, 'pass_att1'] = 1
dg['pass_att2'] = 0
dg.loc[dg['attention']==0.5, 'pass_att2'] = 1

#### Confidence

In [43]:
#calculate weighted average
dg['wa_confidence'] = np.nan
for i in range(dg.shape[0]):
    wa = 0
    for j in range(11):
        wa += (dg['confidence' + str(j)][i]/100)*j
    dg.loc[i, 'wa_confidence'] = wa

In [44]:
dg['wa_confidence'].describe()

count    27.000000
mean      4.882593
std       2.296014
min       0.920000
25%       3.415000
50%       4.720000
75%       5.880000
max       9.450000
Name: wa_confidence, dtype: float64

In [45]:
# overconfidence within subject
dg['overconfidence'] = dg['wa_confidence']-dg['participant.overall_score']
dg['overconfidence'].describe()

count    27.000000
mean      2.030741
std       2.491759
min      -2.230000
25%      -0.175000
50%       2.290000
75%       3.285000
max       6.970000
Name: overconfidence, dtype: float64

In [46]:
#calculate weighted average
dg['wa_difficulty'] = np.nan
for i in range(dg.shape[0]):
    wa = 0
    for j in range(11):
        wa += (dg['difficulty' + str(j)][i]/100)*j
    dg.loc[i, 'wa_difficulty'] = wa

In [47]:
dg['wa_difficulty'].describe()

count    27.000000
mean      4.688148
std       1.625182
min       0.760000
25%       4.110000
50%       4.850000
75%       5.445000
max       9.090000
Name: wa_difficulty, dtype: float64

In [48]:
# overconfidence compared to others
dg['overconfidence_to_others'] = dg['wa_confidence']-dg['wa_difficulty']
dg['overconfidence_to_others'].describe()

count    27.000000
mean      0.194444
std       1.459413
min      -3.540000
25%      -0.540000
50%       0.140000
75%       0.985000
max       2.960000
Name: overconfidence_to_others, dtype: float64

In [49]:
dg['delegation'].mean()

0.4074074074074074

In [50]:
dg[['delegation', 'belief_del_good', 'belief_del_bad', 'belief_nodel_good', 'belief_nodel_bad', 'random_order_del', 'switching_point']].corr()

,delegation,belief_del_good,belief_del_bad,belief_nodel_good,belief_nodel_bad,random_order_del,switching_point
delegation,1.000000,-0.045740,0.032505,0.255753,0.206846,0.232955,0.107891
belief_del_good,-0.045740,1.000000,0.174259,0.680499,0.110410,0.302590,-0.033146
belief_del_bad,0.032505,0.174259,1.000000,0.056968,0.468348,0.192709,0.150587
belief_nodel_good,0.255753,0.680499,0.056968,1.000000,0.023079,0.173628,-0.141706
belief_nodel_bad,0.206846,0.110410,0.468348,0.023079,1.000000,0.242044,0.004233
random_order_del,0.232955,0.302590,0.192709,0.173628,0.242044,1.000000,-0.156932
switching_point,0.107891,-0.033146,0.150587,-0.141706,0.004233,-0.156932,1.000000


In [51]:
pd.crosstab(dg['delegation'], dg['random_order_del'])

random_order_del,0,1
delegation,,
0,11,5
1,5,6


In [52]:
dg[['delegation', 'wa_confidence', 'overconfidence', 'participant.overall_score', 'overconfidence_to_others']].corr()

,delegation,wa_confidence,overconfidence,participant.overall_score,overconfidence_to_others
delegation,1.000000,0.038523,0.077741,-0.079653,0.202170
wa_confidence,0.038523,1.000000,0.850718,0.133354,0.710324
overconfidence,0.077741,0.850718,1.000000,-0.407482,0.579347
participant.overall_score,-0.079653,0.133354,-0.407482,1.000000,0.141744
overconfidence_to_others,0.202170,0.710324,0.579347,0.141744,1.000000


#### Punishment Beliefs

In [53]:
dg[['belief_del_good', 'belief_del_bad', 'belief_nodel_good', 'belief_nodel_bad']].describe()

,belief_del_good,belief_del_bad,belief_nodel_good,belief_nodel_bad
count,27.000000,27.000000,27.000000,27.000000
mean,0.755556,0.983333,0.607407,0.985185
std,0.606429,0.551397,0.561198,0.654689
min,0.000000,0.000000,0.000000,0.000000
25%,0.225000,0.700000,0.075000,0.475000
50%,1.000000,1.000000,0.550000,1.000000
75%,1.000000,1.050000,1.000000,1.550000
max,2.000000,2.000000,2.000000,2.000000


In [54]:
#create var for diff in beliefs
dg['bad_good_del'] = dg['belief_del_bad'] - dg['belief_del_good']
dg['bad_good_nodel'] = dg['belief_nodel_bad'] - dg['belief_nodel_good']
dg['nodel_del_good'] = dg['belief_nodel_good'] - dg['belief_del_good']
dg['nodel_del_bad'] = dg['belief_nodel_bad'] - dg['belief_del_bad']

In [55]:
dg[['bad_good_del', 'bad_good_nodel', 'nodel_del_good', 'nodel_del_bad']].describe()

,bad_good_del,bad_good_nodel,nodel_del_good,nodel_del_bad
count,27.000000,27.000000,27.000000,27.000000
mean,0.227778,0.377778,-0.148148,0.001852
std,0.745155,0.852410,0.468525,0.628105
min,-1.000000,-0.850000,-1.000000,-1.500000
25%,-0.275000,-0.100000,-0.500000,-0.225000
50%,0.000000,0.000000,0.000000,0.000000
75%,0.800000,0.650000,0.025000,0.225000
max,2.000000,2.000000,0.750000,1.500000


In [56]:
dg[['delegation', 'bad_good_del', 'bad_good_nodel', 'nodel_del_good', 'nodel_del_bad']].corr()

,delegation,bad_good_del,bad_good_nodel,nodel_del_good,nodel_del_bad
delegation,1.000000,0.061278,-0.009512,0.365543,0.187065
bad_good_del,0.061278,1.000000,0.534022,0.273609,-0.257532
bad_good_nodel,-0.009512,0.534022,1.000000,-0.297232,0.501859
nodel_del_good,0.365543,0.273609,-0.297232,1.000000,0.017959
nodel_del_bad,0.187065,-0.257532,0.501859,0.017959,1.000000


#### Risk Prefs

In [57]:
dg['switching_point'].describe()

count     27.000000
mean      67.407407
std       20.303958
min        0.000000
25%       60.000000
50%       70.000000
75%       80.000000
max      100.000000
Name: switching_point, dtype: float64

#### Questionnaire

In [58]:
r = [print(i) for i in dg['research_about']]

I think the research was to see how different people react to rewards like this when shared with other people. 
probablity
Peoples perception of size and maybe their ability to guess?
Probability
dont know
I really don't know
I'm not totally sure but I enjoyed it
probability of predicting and whether delegating to ai affects our choices
no idea.
Assuming how much a person weighs depending on their profile pic
How risk averse I am.
I think it's about people's ability to make correct estimations.
how much confidence people would have in their impressions
Peoples judgement and confidence in their estimations 
weather people choose better than an algorithm maybe
trust in algorithms and risk taking
In hindsight I think the performance of the algorithm vs my performance was random and that's what you were really testing
To see if people are more prone to delegating tasks than others.
facial predictions of size
How much we trust AI rather than 'real' people.
Not sure 
Risk assessment 
Predict

In [59]:
r = [print(i) for i in dg['del_reason']]

There's no particular reason, to be honest. 
im sure it would give the best possible answer
As my choices were already similar to the algorithm, I felt I had more control in the further tests with my eyes and information given. 
I am a pretty good judge of people's weight, so I trusted my own instincs
would rather trust myself than a machine
I didn't have any great confidence in my own answers and as the algorithm was no worse decided to hand responsibility over rather than be personally responsible for costing somebody else.
I thought I would do a better job than the algorithm
i thought i could predict better more likely
Thats the way to go. AI
I didn't see a reason why I should
I think I'm good at estimating weights.
Because i think an algorithm is more likely to predict the correct answer
would be more informed then I am
I trusted my ability to make an accurate estimate 
Algorithm might be smarter and make better decision
I just thought it would be better than me at making decisions

In [60]:
r = [print(i) for i in dg['unclear']]

No, it was pretty easy to understand. 
no
No
The final part about choosing a column. At first I wasn't sure if I had to choose the whole secion of a letter.
no
Not at all, perfectly clear thank you
No
none
no
No
I doubt there is really a 'partner' at all.
No
No
No
no
no
all looked clear. The probability distribution bits were a bit clunky to adjust, too many sliders and difficult to get the sum to 100%
No
a little unclear on last choice - % choice of £1 or 50p for sure
No
All seemed clear
The last question it was slightly unclear if choosing option a or b in tick boxes 
No
Nothing was unclear. :)
No
No
Complicated explanations


## Regressions

In [104]:
#do regressions for dg, jeweils bereinigt von attention checks und multipliziert auf 100 obs
dg_att = dg[dg['attention']==0.5]

dg_3 = pd.DataFrame(np.repeat(dg.values, 3, axis=0), columns=dg.columns)
dg_att_3 = pd.DataFrame(np.repeat(dg_att.values, 3, axis=0), columns=dg_att.columns)

In [105]:
dg['nodel_del_good'].mean()

-0.1481481481481481

In [106]:
dg_att['nodel_del_good'].mean()

-0.17941176470588238

In [107]:
dg_4['nodel_del_good'].mean()

-0.14814814814814806

In [108]:
dg_att_6['nodel_del_good'].mean()

-0.17941176470588235

In [122]:
est = []

for d in [dg, dg_att, dg_3, dg_att_3]:
    
    param_specs = [
        ['wa_confidence'],
        ['switching_point'],
        ['nodel_del_good', 'nodel_del_bad'],
        ['wa_confidence', 'switching_point', 'nodel_del_good', 'nodel_del_bad', 'Age', 'female']
    ]

    
    for i in param_specs:
        l = i.copy()
        l.append('delegation')

        reg = d[l].copy()
        reg = reg.dropna()
        Y = reg['delegation']
        X = sm.add_constant(reg[i])
        model = sm.OLS(Y.astype(float),X.astype(float)).fit()
        est.append(model)
stargazer = Stargazer(est)
stargazer.custom_columns(['All observations', 'passed attention check', 'All observations (3x)', 'passed attention check (3x)'], [4, 4, 4, 4])
stargazer.significant_digits(2)
stargazer.covariate_order(['const', 'nodel_del_good', 'nodel_del_bad', 'Age', 'female', 'switching_point', 'wa_confidence'])
stargazer.rename_covariates({'nodel_del_good': 'good (no del-del)', 'nodel_del_bad': 'bad (no del-del)', 'switching_point': 'risk aversion'})
stargazer.show_degrees_of_freedom(False)

stargazer

C:\Users\felix\anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)


In [123]:
print(stargazer.render_latex())

\begin{table}[!htbp] \centering
\begin{tabular}{@{\extracolsep{5pt}}lcccccccccccccccc}
\\[-1.8ex]\hline
\hline \\[-1.8ex]
& \multicolumn{16}{c}{\textit{Dependent variable:}} \
\cr \cline{16-17}
\\[-1.8ex] & \multicolumn{4}{c}{All observations} & \multicolumn{4}{c}{passed attention check} & \multicolumn{4}{c}{All observations (3x)} & \multicolumn{4}{c}{passed attention check (3x)}  \\
\\[-1.8ex] & (1) & (2) & (3) & (4) & (5) & (6) & (7) & (8) & (9) & (10) & (11) & (12) & (13) & (14) & (15) & (16) \\
\hline \\[-1.8ex]
 const & 0.37$^{}$ & 0.23$^{}$ & 0.46$^{***}$ & 0.35$^{}$ & -0.23$^{}$ & 0.30$^{}$ & 0.45$^{***}$ & -1.18$^{}$ & 0.37$^{***}$ & 0.23$^{}$ & 0.46$^{***}$ & 0.35$^{}$ & -0.23$^{}$ & 0.30$^{}$ & 0.45$^{***}$ & -1.18$^{***}$ \\
  & (0.23) & (0.34) & (0.10) & (0.64) & (0.35) & (0.65) & (0.11) & (0.84) & (0.13) & (0.19) & (0.05) & (0.33) & (0.19) & (0.36) & (0.06) & (0.40) \\
 good (no del-del) & & & 0.39$^{*}$ & 0.44$^{*}$ & & & 0.53$^{**}$ & 0.40$^{**}$ & & & 0.39$^{***}$ & 0.4

In [132]:
pd.crosstab(dg_att['female'], dg_att['delegation'])

delegation,0,1
female,,
0.0,9,1
1.0,2,5


In [136]:
from statsmodels.discrete.discrete_model import Probit

d = dg_att.copy()
param_specs = [
        ['wa_confidence'],
        ['switching_point'],
        ['nodel_del_good', 'nodel_del_bad'],
        ['wa_confidence', 'switching_point', 'nodel_del_good', 'nodel_del_bad', 'Age', 'female']
]


for i in param_specs:
    l = i.copy()
    l.append('delegation')

    reg = d[l].copy()
    reg = reg.dropna()
    Y = reg['delegation']
    X = sm.add_constant(reg[i])
    model = Probit(Y.astype(float),X.astype(float)).fit()
    print(model.summary())

Optimization terminated successfully.
         Current function value: 0.553391
         Iterations 6
                          Probit Regression Results                           
Dep. Variable:             delegation   No. Observations:                   17
Model:                         Probit   Df Residuals:                       15
Method:                           MLE   Df Model:                            1
Date:                Mon, 06 Mar 2023   Pseudo R-squ.:                  0.1476
Time:                        12:11:58   Log-Likelihood:                -9.4076
converged:                       True   LL-Null:                       -11.037
Covariance Type:            nonrobust   LLR p-value:                   0.07103
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const            -2.3438      1.273     -1.841      0.066      -4.839       0.152
wa_confidence     0.

C:\Users\felix\anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
C:\Users\felix\anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
C:\Users\felix\anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)
C:\Users\felix\anaconda3\lib\site-packages\statsmodels\tsa\tsatools.py:142: FutureWarning: In a future version of pandas all arguments of concat except for the argument 'objs' will be keyword-only
  x = pd.concat(x[::order], 1)


PerfectSeparationError: Perfect separation detected, results not available

In [143]:
import scipy
for d in [dg, dg_att, dg_3, dg_att_3]:
    print('d')
    for var in ['wa_confidence', 'switching_point', 'nodel_del_good', 'nodel_del_bad', 'Age', 'female']:
        print(var)
        print(scipy.stats.mannwhitneyu(d[d['delegation']==1][var].astype(float).dropna(), d[d['delegation']==0][var].astype(float).dropna()))

d
wa_confidence
MannwhitneyuResult(statistic=90.0, pvalue=0.9409948168019828)
switching_point
MannwhitneyuResult(statistic=93.5, pvalue=0.8022864712387354)
nodel_del_good
MannwhitneyuResult(statistic=128.0, pvalue=0.04540884380615528)
nodel_del_bad
MannwhitneyuResult(statistic=103.0, pvalue=0.47333937069543086)
Age
MannwhitneyuResult(statistic=95.5, pvalue=0.5159979314357037)
female
MannwhitneyuResult(statistic=94.5, pvalue=0.4897423219193927)
d
wa_confidence
MannwhitneyuResult(statistic=44.0, pvalue=0.3010665804783452)
switching_point
MannwhitneyuResult(statistic=34.0, pvalue=0.9590865396148249)
nodel_del_good
MannwhitneyuResult(statistic=55.0, pvalue=0.0249627966988456)
nodel_del_bad
MannwhitneyuResult(statistic=38.0, pvalue=0.6504762388917988)
Age
MannwhitneyuResult(statistic=50.5, pvalue=0.0869494397079922)
female
MannwhitneyuResult(statistic=54.5, pvalue=0.01344882866887079)
d
wa_confidence
MannwhitneyuResult(statistic=810.0, pvalue=0.8663396473997563)
switching_point
Mannwhitneyu

In [144]:
pd.crosstab(dg['Age'], dg['delegation'])

delegation,0,1
Age,,
20.0,1,0
21.0,0,1
26.0,0,1
27.0,2,0
30.0,1,0
31.0,1,1
32.0,1,1
33.0,1,0
34.0,2,0


#### Prolific Metadata

In [134]:
dg['Total approvals'].describe()

count      26.000000
mean      390.230769
std       313.743246
min         6.000000
25%       142.500000
50%       293.000000
75%       590.750000
max      1049.000000
Name: Total approvals, dtype: float64

In [135]:
dg['Time taken'].describe()

count      26.000000
mean      713.269231
std       280.737394
min       392.000000
25%       520.250000
50%       649.500000
75%       818.250000
max      1669.000000
Name: Time taken, dtype: float64

In [136]:
dg[['Total approvals', 'Time taken']].corr()

,Total approvals,Time taken
Total approvals,1.000000,-0.096063
Time taken,-0.096063,1.000000


In [145]:
dg['pass'] = 0
dg.loc[dg['attention']==0.5, 'pass'] = 1

In [147]:
dg[['pass', 'Time taken', 'Total approvals']].corr()

,pass,Time taken,Total approvals
pass,1.000000,0.253568,-0.161066
Time taken,0.253568,1.000000,-0.096063
Total approvals,-0.161066,-0.096063,1.000000


In [29]:
#check for connection between time taken and attention tests
dg[['Total approvals', 'Time taken', 'attention']]

,Total approvals,Time taken,attention
0,NaN,NaN,NaN
1,358.0,735.0,NaN
2,6.0,530.0,0.50
3,984.0,968.0,1.00
4,564.0,652.0,2.00
5,652.0,894.0,0.50
6,240.0,496.0,0.50
7,743.0,593.0,0.50
8,312.0,455.0,0.75
9,312.0,392.0,0.30


In [100]:
[print(i) for i in dg.columns]

Unnamed: 0
participant.id_in_session
participant.code
participant.overall_score
participant.relevant_round
session.code
bad_outcome_eva
good_outcome_eva
bad_outcome_del
good_outcome_del
max_punishment
suf
punishment_cost
n_preds
belief_payment_confidence
belief_payment_difficulty
risk_payment
margin_of_error
intro.id_in_group
test_slider
guess_r1
truth_r1
success_r1
guess_r2
truth_r2
success_r2
guess_r3
truth_r3
success_r3
guess_r4
truth_r4
success_r4
guess_r5
truth_r5
success_r5
guess_r6
truth_r6
success_r6
guess_r7
truth_r7
success_r7
guess_r8
truth_r8
success_r8
guess_r9
truth_r9
success_r9
guess_r10
truth_r10
success_r10
post_decisions.id_in_group
confidence0
confidence1
confidence2
confidence3
confidence4
confidence5
confidence6
confidence7
confidence8
confidence9
confidence10
error_confidence
pay_belief
difficulty0
difficulty1
difficulty2
difficulty3
difficulty4
difficulty5
difficulty6
difficulty7
difficulty8
difficulty9
difficulty10
error_difficulty
random_order_del
delegation
g

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None]